# 00 - Données, géodésie et orthophotos

Ce notebook contrôle la configuration du projet, explicite les CRS utilisés et prépare les échantillons WMTS SWISSIMAGE pour les cinq agglomérations.


In [1]:
# Paramètres
from pathlib import Path

PROJECT_ROOT = Path.cwd()
ORTHO_PROFILES = ["swissimage_25cm_like", "swissimage_10cm_like"]
ORTHO_MAX_TILES_PER_PROFILE = 10

CRS_LV95 = "EPSG:2056"
CRS_WGS84 = "EPSG:4326"
CRS_WMTS = "EPSG:3857"

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Exécuter ce notebook depuis la racine du dépôt projet-slow-vaud.")


In [2]:
import json
import sys

import pandas as pd

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from slowvaud.crs import wgs84_to_lv95
from slowvaud.orthophotos import download_records, iter_tile_records, write_manifest
from slowvaud.paths import data_paths, ensure_data_tree, load_config


## Configuration et centres d'agglomération

Les centres sont stockés en WGS84 dans la configuration. Les coordonnées LV95 ci-dessous servent au contrôle et aux traitements métriques.


In [3]:
cfg = load_config()
paths = ensure_data_tree()

rows = []
for agglo_key, agglo in cfg["agglomerations"].items():
    lon = float(agglo["center_wgs84"]["lon"])
    lat = float(agglo["center_wgs84"]["lat"])
    east, north = wgs84_to_lv95(lon, lat)
    rows.append({
        "agglo_key": agglo_key,
        "label": agglo["label"],
        "lon_wgs84": lon,
        "lat_wgs84": lat,
        "east_lv95": round(east, 2),
        "north_lv95": round(north, 2),
        "ortho_buffer_m": agglo["ortho_buffer_m"],
    })

pd.DataFrame(rows)


,agglo_key,label,lon_wgs84,lat_wgs84,east_lv95,north_lv95,ortho_buffer_m
0,lausanne,Lausanne,6.6323,46.5197,2538124.47,1152363.14,2500
1,berne,Berne,7.4474,46.9479,2600667.48,1199646.20,2500
2,geneve,Genève,6.1432,46.2044,2500016.02,1117821.07,2500
3,bale,Bâle,7.5886,47.5596,2611287.84,1267664.85,2500
4,zurich,Zurich,8.5472,47.3769,2683719.22,1247931.48,2500


## Contrat CRS


In [4]:
crs_rows = [
    {"objet": "agglomérations GeoAdmin", "crs": CRS_LV95, "usage": "périmètres bruts et dissous"},
    {"objet": "centres de configuration", "crs": CRS_WGS84, "usage": "points lon/lat"},
    {"objet": "OSM et GeoJSON web", "crs": CRS_WGS84, "usage": "Overpass, SITG GeoJSON"},
    {"objet": "tuiles WMTS GeoAdmin", "crs": CRS_WMTS, "usage": "indices XYZ"},
]
pd.DataFrame(crs_rows)


,objet,crs,usage
0,agglomérations GeoAdmin,EPSG:2056,périmètres bruts et dissous
1,centres de configuration,EPSG:4326,points lon/lat
2,OSM et GeoJSON web,EPSG:4326,"Overpass, SITG GeoJSON"
3,tuiles WMTS GeoAdmin,EPSG:3857,indices XYZ


## Manifeste et téléchargement WMTS

Le manifeste liste un échantillon limité par agglomération et par profil. Les fichiers existants ne sont pas retéléchargés.


In [5]:
records = iter_tile_records(
    agglomerations=list(cfg["agglomerations"].keys()),
    profiles=ORTHO_PROFILES,
    max_tiles_per_profile=ORTHO_MAX_TILES_PER_PROFILE,
)

manifest_path = paths["manifests"] / "orthophoto_wmts_manifest.csv"
write_manifest(records, manifest_path)

stats = download_records(records, dry_run=False)
stats_path = paths["manifests"] / "orthophoto_download_stats.json"
stats_path.write_text(json.dumps(stats, indent=2, ensure_ascii=False), encoding="utf-8")

summary = (
    pd.DataFrame(stats)
    .groupby(["agglomeration", "profile", "status"], as_index=False)
    .agg(files=("output", "count"), bytes=("bytes", "sum"))
)

print(f"Manifeste: {manifest_path}")
print(f"Statistiques: {stats_path}")
summary


Manifeste: /Users/marc-ed/BAS-local/repos/projet-slow-vaud/data/manifests/orthophoto_wmts_manifest.csv
Statistiques: /Users/marc-ed/BAS-local/repos/projet-slow-vaud/data/manifests/orthophoto_download_stats.json


,agglomeration,profile,status,files,bytes
0,bale,swissimage_10cm_like,downloaded,1,3939
1,bale,swissimage_10cm_like,exists,9,72797
2,bale,swissimage_25cm_like,downloaded,1,7202
3,bale,swissimage_25cm_like,exists,9,146431
4,berne,swissimage_10cm_like,exists,10,197740
5,berne,swissimage_25cm_like,exists,10,220987
6,geneve,swissimage_10cm_like,downloaded,10,198074
7,geneve,swissimage_25cm_like,downloaded,1,24225
8,geneve,swissimage_25cm_like,exists,9,219944
9,lausanne,swissimage_10cm_like,downloaded,10,192258


In [6]:
manifest = pd.read_csv(manifest_path)
required_crs_columns = {"bbox_crs", "wmts_crs"}
missing_columns = required_crs_columns - set(manifest.columns)
if missing_columns:
    raise ValueError(f"Colonnes CRS manquantes dans le manifeste: {sorted(missing_columns)}")

if set(manifest["bbox_crs"]) != {CRS_WGS84}:
    raise ValueError("Le manifeste orthophoto doit déclarer bbox_crs = EPSG:4326.")
if set(manifest["wmts_crs"]) != {CRS_WMTS}:
    raise ValueError("Le manifeste orthophoto doit déclarer wmts_crs = EPSG:3857.")

manifest.head()


,agglomeration,agglomeration_label,profile,profile_label,layer,zoom,tile_x,tile_y,extension,url,center_lon,center_lat,buffer_m,bbox_wgs84,bbox_crs,wmts_crs,estimated_pixel_size_m,tiles_in_full_bbox,manifest_limited
0,lausanne,Lausanne,swissimage_25cm_like,Echantillonnage WMTS proche 20-25 cm,ch.swisstopo.swissimage,19,271755,185379,jpeg,https://wmts.geo.admin.ch/1.0.0/ch.swisstopo.s...,6.6323,46.5197,2500.0,"[6.599723329976529, 46.497212192365055, 6.6648...",EPSG:4326,EPSG:3857,0.2055,9216,True
1,lausanne,Lausanne,swissimage_25cm_like,Echantillonnage WMTS proche 20-25 cm,ch.swisstopo.swissimage,19,271755,185380,jpeg,https://wmts.geo.admin.ch/1.0.0/ch.swisstopo.s...,6.6323,46.5197,2500.0,"[6.599723329976529, 46.497212192365055, 6.6648...",EPSG:4326,EPSG:3857,0.2055,9216,True
2,lausanne,Lausanne,swissimage_25cm_like,Echantillonnage WMTS proche 20-25 cm,ch.swisstopo.swissimage,19,271755,185381,jpeg,https://wmts.geo.admin.ch/1.0.0/ch.swisstopo.s...,6.6323,46.5197,2500.0,"[6.599723329976529, 46.497212192365055, 6.6648...",EPSG:4326,EPSG:3857,0.2055,9216,True
3,lausanne,Lausanne,swissimage_25cm_like,Echantillonnage WMTS proche 20-25 cm,ch.swisstopo.swissimage,19,271755,185382,jpeg,https://wmts.geo.admin.ch/1.0.0/ch.swisstopo.s...,6.6323,46.5197,2500.0,"[6.599723329976529, 46.497212192365055, 6.6648...",EPSG:4326,EPSG:3857,0.2055,9216,True
4,lausanne,Lausanne,swissimage_25cm_like,Echantillonnage WMTS proche 20-25 cm,ch.swisstopo.swissimage,19,271755,185383,jpeg,https://wmts.geo.admin.ch/1.0.0/ch.swisstopo.s...,6.6323,46.5197,2500.0,"[6.599723329976529, 46.497212192365055, 6.6648...",EPSG:4326,EPSG:3857,0.2055,9216,True
